Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public
License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later
version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied
warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see
<https://www.gnu.org/licenses/>.

# H11 v5: Sharp Basins From Binary Keys, a Pilot

*A hunch* ([docs/hypotheses/README.md § H11](../docs/hypotheses/README.md#h11-a-hebbian-hopfield-transformer)): one
transformer block whose memory is a Hopfield matrix written by Hebb's rule as the tokens arrive and read by noisy,
temperature-controlled mean-field settling, the whole trained by backpropagation. In four tests (v1 to v4, their
notebooks beside this one) the Hopfield settling never beat its own ablation. A hunch from them: with continuous keys
of low gain the settling has no sharp basins, so it cannot snap a noisy cue onto one stored key. v5 tests that in the
cheapest way: v4's model with its keys binarised to +1 and -1 (a straight-through estimator: the sign forward, the
gradient of `tanh` backward), so that the stored keys are corners of the cube and the settling has corners to fall
into. A pilot, pre-registered in the 2026-10-14 journal (*H11 v5, Sharp Basins From Binary Keys: A Pilot*) before it
was run, on v3's noisy-cue task.

The task is associative recall after [Ba et al. 2016](https://arxiv.org/abs/1610.06258). This explores; nothing the
benchmark depends on lives here.

In [1]:
import math, os, time
import multiprocessing as mp
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
matplotlib.use("svg")
import matplotlib.pyplot as plt

print("torch", torch.__version__, "numpy", np.__version__, "matplotlib", matplotlib.__version__)

torch 2.14.1+cpu numpy 2.5.4 matplotlib 3.11.2


## The Task

Sixteen key-value pairs, keys drawn without repetition from 128 key tokens and values from 16 value tokens, then a
query mark and one of the sixteen keys; the answer is that key's value. Chance is 1/16.

In [2]:
NK, NV, NPAIR = 128, 16, 16          # key vocabulary, value vocabulary, pairs per sequence
QUERY = NK + NV                      # the query mark
VOCAB = NK + NV + 1
DM = 64                              # model width


def make_batch(gen, b, n=NPAIR):
    keys = torch.stack([torch.randperm(NK, generator=gen)[:n] for _ in range(b)])
    vals = torch.randint(0, NV, (b, n), generator=gen)
    pick = torch.randint(0, n, (b,), generator=gen)
    seq = torch.empty(b, 2 * n + 2, dtype=torch.long)
    seq[:, 0:2 * n:2] = keys
    seq[:, 1:2 * n:2] = NK + vals
    seq[:, -2] = QUERY
    seq[:, -1] = keys[torch.arange(b), pick]
    return seq, vals[torch.arange(b), pick]


def answer_by_reading(seq):
    """The answer found by reading the sequence, independently of how make_batch built it."""
    out = []
    for row in seq.tolist():
        q = row[-1]
        out.append(next(row[i + 1] - NK for i in range(0, len(row) - 2, 2) if row[i] == q))
    return torch.tensor(out)


s, y = make_batch(torch.Generator().manual_seed(0), 4)
print(s[0].tolist(), "->", y[0].item())

[44, 134, 94, 138, 121, 133, 13, 128, 31, 142, 116, 139, 71, 128, 84, 138, 105, 140, 14, 131, 15, 138, 22, 129, 48, 138, 11, 137, 30, 140, 77, 141, 144, 13] -> 0


## The Models

One block each: an embedding, normalised to length `sqrt(64)` so that a model cannot outgrow the noise by scaling
it, with Gaussian noise of standard deviation `cue_sigma` per dimension added to the query key's embedding alone; a
one-token shift (each position sees its token and the one before), a memory
mixer read at the last position, a residual, layer norm, an MLP and a readout. Only the mixer differs.

- `NoMemory`: the control, which sees only the query.
- `Softmax`: one head of softmax attention, a modern Hopfield step ([Ramsauer et al. 2020](https://arxiv.org/abs/2008.02217)).
- `Hebbian`: linear attention as a fast-weight memory `M <- lam M + g v k^T`
  ([Schlag, Irie and Schmidhuber 2021](https://arxiv.org/abs/2102.11174)), read once.
- `HebbHopfield`, the hybrid: patterns `p = tanh([k; v])` written by the same rule into a symmetric Hopfield matrix
  `W <- lam W + g p p^T / D` ([Hopfield 1982](https://doi.org/10.1073/pnas.79.8.2554)), read by `K` synchronous
  steps of `xi <- tanh(beta W xi + alpha xi0 + sigma eps)` from `xi0 = [tanh(q); 0]`: mean-field settling at inverse
  temperature `beta`, with Gaussian noise in training standing in for a Boltzmann machine's sampling
  ([Ackley, Hinton and Sejnowski 1985](https://doi.org/10.1207/s15516709cog0901_7)). It is a stand-in, not a
  Boltzmann machine: the units are continuous and nothing is sampled at test time.

This notebook runs three of them: `Delta`, DeltaNet's memory `M <- lam M + g (v - M k) k^T`
([Schlag, Irie and Schmidhuber 2021](https://arxiv.org/abs/2102.11174)) with keys of 64 (1024 cells); and `KeyClean`,
v4, with keys of 24: a DeltaNet memory of 24 x 16 and a delta-written Hopfield memory of the keys, 24 x 24 (960 cells
in all), whose settling cleans the cue before the read, here with binary keys - in five steps, and in none. The other classes are defined because the checks and earlier code reuse them.

Because a decaying sum of outer products is linear in its writes, both memories are computed as one weighted sum
rather than a loop over time; the first check below holds the two equal.

In [3]:
class Block(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, DM)
        self.mixer = mixer
        self.norm = nn.LayerNorm(DM)
        self.mlp = nn.Sequential(nn.Linear(DM, 4 * DM), nn.GELU(), nn.Linear(4 * DM, DM))
        self.out = nn.Linear(DM, NV)

    cue_sigma, cue_gen = 0.0, None

    def features(self, seq):
        e = F.normalize(self.emb(seq), dim=-1) * math.sqrt(DM)
        if self.cue_sigma > 0:
            noise = torch.randn(e[:, -1].shape, generator=self.cue_gen)
            e = torch.cat([e[:, :-1], (e[:, -1] + self.cue_sigma * noise)[:, None]], 1)
        prev = torch.cat([torch.zeros_like(e[:, :1]), e[:, :-1]], 1)
        return e, torch.cat([e, prev], -1)

    def forward(self, seq):
        e, x = self.features(seq)
        h = e[:, -1] + self.mixer(x)
        h = h + self.mlp(self.norm(h))
        return self.out(h)


class NoMemory(nn.Module):
    def __init__(self):
        super().__init__()
        self.o = nn.Linear(2 * DM, DM)

    def forward(self, x):
        return self.o(x[:, -1])


class Softmax(nn.Module):
    def __init__(self, dk, dv):
        super().__init__()
        self.dk, self.dv = dk, dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.o = nn.Linear(dv, DM)

    def forward(self, x):
        q, k, v = self.q(x[:, -1]), self.k(x[:, :-1]), self.v(x[:, :-1])
        a = torch.softmax((k @ q.unsqueeze(-1)).squeeze(-1) / math.sqrt(self.dk), -1)
        return self.o((a.unsqueeze(-1) * v).sum(1))


def hebbian_sum(lam, g, a, b):
    """sum_s lam^(T-1-s) g_s a_s b_s^T: the state after the loop m <- lam m + g_s a_s b_s^T over s = 0 .. T-1."""
    t = a.shape[1]
    decay = lam ** torch.arange(t - 1, -1, -1, dtype=a.dtype)
    return torch.einsum("bs,bsi,bsj->bij", g * decay, a, b)


class Hebbian(nn.Module):
    def __init__(self, dk, dv):
        super().__init__()
        self.dk, self.dv = dk, dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.g = nn.Linear(2 * DM, 1)
        self.lam_logit = nn.Parameter(torch.tensor(4.0))
        self.o = nn.Linear(dv, DM)

    def forward(self, x):
        k = F.normalize(self.k(x[:, :-1]), dim=-1)
        v, g = self.v(x[:, :-1]), torch.sigmoid(self.g(x[:, :-1]))[..., 0]
        m = hebbian_sum(torch.sigmoid(self.lam_logit), g, v, k)
        q = F.normalize(self.q(x[:, -1]), dim=-1)
        return self.o((m @ q.unsqueeze(-1)).squeeze(-1))


class HebbHopfield(nn.Module):
    def __init__(self, dk, dv, steps=5, sigma=0.1, learn_beta=True):
        super().__init__()
        self.dk, self.dv, self.steps, self.sigma = dk, dv, steps, sigma
        self.d = dk + dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.g = nn.Linear(2 * DM, 1)
        self.lam_logit = nn.Parameter(torch.tensor(4.0))
        self.log_beta = nn.Parameter(torch.tensor(0.0), requires_grad=learn_beta)
        self.log_alpha = nn.Parameter(torch.tensor(0.0))
        self.o = nn.Linear(dv, DM)

    def settle(self, x):
        """The memory, the start state and every state of the settling, for the read and for diagnostics."""
        b = x.shape[0]
        p = torch.tanh(torch.cat([self.k(x[:, :-1]), self.v(x[:, :-1])], -1))
        g = torch.sigmoid(self.g(x[:, :-1]))[..., 0]
        w = hebbian_sum(torch.sigmoid(self.lam_logit), g, p, p) / self.d
        xi0 = torch.cat([torch.tanh(self.q(x[:, -1])), x.new_zeros(b, self.dv)], -1)
        beta, alpha = self.log_beta.exp(), self.log_alpha.exp()
        states = [xi0]
        for _ in range(self.steps):
            h = beta * (w @ states[-1].unsqueeze(-1)).squeeze(-1) + alpha * xi0
            if self.training and self.sigma > 0:
                h = h + self.sigma * torch.randn_like(h)
            states.append(torch.tanh(h))
        return w, states

    def forward(self, x):
        _, states = self.settle(x)
        return self.o(states[-1][:, self.dk:])


class Delta(nn.Module):
    """DeltaNet's fast weights: M <- lam M + g (v - M k) k^T, written by the error, read once."""

    def __init__(self, dk, dv):
        super().__init__()
        self.dk, self.dv = dk, dv
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.g = nn.Linear(2 * DM, 1)
        self.lam_logit = nn.Parameter(torch.tensor(4.0))
        self.o = nn.Linear(dv, DM)

    def forward(self, x):
        b, t, _ = x.shape
        k = F.normalize(self.k(x[:, :-1]), dim=-1)
        v, g = self.v(x[:, :-1]), torch.sigmoid(self.g(x[:, :-1]))
        lam = torch.sigmoid(self.lam_logit)
        m = x.new_zeros(b, self.dv, self.dk)
        for s in range(t - 1):
            err = v[:, s] - (m @ k[:, s, :, None])[..., 0]
            m = lam * m + g[:, s, :, None] * err[:, :, None] * k[:, s, None, :]
        q = F.normalize(self.q(x[:, -1]), dim=-1)
        return self.o((m @ q[..., None])[..., 0])


class HebbHopfield2(nn.Module):
    """H11 v2: heads of autoassociative Hopfield memories written by the delta (pseudo-inverse) rule
    W <- lam W + g (p - W p) p^T / |p|^2, read by settling with a learned inverse temperature per step."""

    def __init__(self, heads, dk, dv, steps=5, sigma=0.1, delta=True, anneal=True):
        super().__init__()
        self.h, self.dk, self.dv, self.steps, self.sigma, self.delta = heads, dk, dv, steps, sigma, delta
        self.d = dk + dv
        self.q, self.k, self.v = (nn.Linear(2 * DM, heads * dk), nn.Linear(2 * DM, heads * dk),
                                  nn.Linear(2 * DM, heads * dv))
        self.g = nn.Linear(2 * DM, heads)
        self.lam_logit = nn.Parameter(torch.full((heads,), 4.0))
        self.log_beta = nn.Parameter(torch.zeros(steps if anneal else 1))
        self.log_alpha = nn.Parameter(torch.zeros(heads))
        self.o = nn.Linear(heads * dv, DM)

    def settle(self, x):
        b, t, _ = x.shape
        H = self.h
        k = self.k(x[:, :-1]).view(b, t - 1, H, self.dk)
        v = self.v(x[:, :-1]).view(b, t - 1, H, self.dv)
        p = torch.tanh(torch.cat([k, v], -1))                               # b, s, H, D
        g = torch.sigmoid(self.g(x[:, :-1]))                                # b, s, H
        lam = torch.sigmoid(self.lam_logit)[None, :, None, None]
        if self.delta:
            w = x.new_zeros(b, H, self.d, self.d)
            for s in range(t - 1):
                ps = p[:, s]
                err = ps - (w @ ps[..., None])[..., 0]
                w = lam * w + (g[:, s, :, None, None] * err[..., None] * ps[..., None, :]
                               / (ps * ps).sum(-1)[..., None, None].clamp_min(1e-6))
        else:
            decay = torch.sigmoid(self.lam_logit)[None, :] ** torch.arange(t - 2, -1, -1, dtype=x.dtype)[:, None]
            w = torch.einsum("bsh,bshi,bshj->bhij", g * decay, p, p) / self.d
        xi0 = torch.cat([torch.tanh(self.q(x[:, -1]).view(b, H, self.dk)), x.new_zeros(b, H, self.dv)], -1)
        alpha = self.log_alpha.exp()[None, :, None]
        states = [xi0]
        for i in range(self.steps):
            beta = self.log_beta[min(i, len(self.log_beta) - 1)].exp()
            h = beta * (w @ states[-1][..., None])[..., 0] + alpha * xi0
            if self.training and self.sigma > 0:
                h = h + self.sigma * torch.randn_like(h)
            states.append(torch.tanh(h))
        return w, states

    def forward(self, x):
        _, states = self.settle(x)
        return self.o(states[-1][..., self.dk:].reshape(x.shape[0], -1))


class KeyClean(nn.Module):
    """H11 v4, a superset of DeltaNet: keys k = tanh(.) written by the delta rule into a heteroassociative memory
    M <- lam M + g (v - M k^) k^T (k^ = k / |k|) and an autoassociative key memory C <- lam C + g (k - C k) k^T / |k|^2;
    the query settles in C, xi <- tanh(beta_s C xi + alpha xi0), before M reads it. With no steps it is DeltaNet."""

    def __init__(self, dk, dv, steps=5, binary=False):
        super().__init__()
        self.dk, self.dv, self.steps, self.binary = dk, dv, steps, binary
        self.q, self.k, self.v = nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dk), nn.Linear(2 * DM, dv)
        self.g = nn.Linear(2 * DM, 1)
        self.lam_logit = nn.Parameter(torch.tensor(4.0))
        self.log_beta = nn.Parameter(torch.zeros(max(steps, 1)))
        self.log_alpha = nn.Parameter(torch.tensor(0.0))
        self.o = nn.Linear(dv, DM)

    def memories(self, x):
        b, t, _ = x.shape
        k = torch.tanh(self.k(x[:, :-1]))
        if self.binary:
            k = k + (torch.sign(k) - k).detach()
        kn = F.normalize(k, dim=-1)
        v, g = self.v(x[:, :-1]), torch.sigmoid(self.g(x[:, :-1]))
        lam = torch.sigmoid(self.lam_logit)
        m = x.new_zeros(b, self.dv, self.dk)
        c = x.new_zeros(b, self.dk, self.dk)
        for s in range(t - 1):
            ks, kns, gs = k[:, s], kn[:, s], g[:, s, :, None]
            m = lam * m + gs * (v[:, s] - (m @ kns[..., None])[..., 0])[..., None] * kns[:, None, :]
            c = lam * c + gs * (ks - (c @ ks[..., None])[..., 0])[..., None] * ks[:, None, :] / (ks * ks).sum(-1)[:, None, None].clamp_min(1e-6)
        return m, c

    def settle(self, x):
        m, c = self.memories(x)
        xi0 = torch.tanh(self.q(x[:, -1]))
        states = [xi0]
        for i in range(self.steps):
            h = self.log_beta[i].exp() * (c @ states[-1][..., None])[..., 0] + self.log_alpha.exp() * xi0
            states.append(torch.tanh(h))
        return m, c, states

    def forward(self, x):
        m, _, states = self.settle(x)
        return self.o((m @ F.normalize(states[-1], dim=-1)[..., None])[..., 0])


MODELS = {
    "delta-wide": lambda: Delta(64, 16),
    "keyclean-bin": lambda: KeyClean(24, 16, binary=True),
    "keyclean-bin-0step": lambda: KeyClean(24, 16, steps=0, binary=True),
}

## Checks Before the Run

Each was seen failing once, deliberately, before it was trusted (the journal says how): the task's answer agrees with
an independent reading of the sequence, the vectorised Hebbian sum agrees with the loop it replaces, and the hybrid's
memory is symmetric, as a Hopfield matrix must be for its energy to exist; and the cue noise reaches the query and
nothing else; the delta writes of `HebbHopfield2`, `Delta` and v4 store their latest pattern exactly; v4 with no settling
reads exactly as DeltaNet does; and binary keys are stored as corners of the cube while a gradient still reaches them.

In [4]:
for n in (NPAIR, 48):
    seq, y = make_batch(torch.Generator().manual_seed(7), 512, n)
    assert torch.equal(answer_by_reading(seq), y), "the task's answer disagrees with the sequence"

gen = torch.Generator().manual_seed(8)
a, b, g = torch.randn(3, 9, 5, generator=gen), torch.randn(3, 9, 4, generator=gen), torch.rand(3, 9, generator=gen)
lam = torch.tensor(0.9)
m = torch.zeros(3, 5, 4)
for s in range(9):
    m = lam * m + g[:, s, None, None] * a[:, s, :, None] * b[:, s, None, :]
assert torch.allclose(hebbian_sum(lam, g, a, b), m, atol=1e-5), "the vectorised Hebbian sum disagrees with the loop"

torch.manual_seed(0)
blk = Block(HebbHopfield(16, 16)).eval()
with torch.no_grad():
    _, x = blk.features(seq[:8])
    w, _ = blk.mixer.settle(x)
assert torch.allclose(w, w.transpose(1, 2)), "the Hopfield matrix is not symmetric"

with torch.no_grad():
    e0, _ = blk.features(seq[:8])
    blk.cue_sigma, blk.cue_gen = 1.0, torch.Generator().manual_seed(1)
    e1, _ = blk.features(seq[:8])
    blk.cue_sigma, blk.cue_gen = 0.0, None
assert torch.equal(e0[:, :-1], e1[:, :-1]), "the cue noise reached a position other than the query"
assert not torch.allclose(e0[:, -1], e1[:, -1]), "the cue noise did not reach the query"

# The delta rule with the gate open and no decay makes the latest pattern an exact fixed point (W p = p) and the
# latest key an exact address (M k = v); Hebb's rule would leave crosstalk from the earlier writes.
x = torch.randn(3, 3, 2 * DM, generator=torch.Generator().manual_seed(9))
hh, dn = HebbHopfield2(1, 4, 4), Delta(8, 4)
with torch.no_grad():
    for mod in (hh, dn):
        mod.g.weight.zero_(); mod.g.bias.fill_(50.0); mod.lam_logit.fill_(50.0)
    w, _ = hh.settle(x)
    p = torch.tanh(torch.cat([hh.k(x[:, 1]), hh.v(x[:, 1])], -1))
    assert torch.allclose((w[:, 0] @ p[..., None])[..., 0], p, atol=1e-4), "the delta write is not a projection"
    k, v = F.normalize(dn.k(x[:, 1]), dim=-1), dn.v(x[:, 1])
    m = x.new_zeros(3, 4, 8)
    for s in range(2):
        ks, vs = F.normalize(dn.k(x[:, s]), dim=-1), dn.v(x[:, s])
        m = m + (vs - (m @ ks[..., None])[..., 0])[..., None] * ks[:, None, :]
    assert torch.allclose((m @ k[..., None])[..., 0], v, atol=1e-4), "the reference delta write is wrong"
    dn.o = nn.Identity()
    q_eq_k = torch.cat([x[:, :2], x[:, 1:2]], 1)
    dn.q = dn.k
    assert torch.allclose(dn(q_eq_k), v, atol=1e-4), "DeltaNet's memory does not return the latest value"

# v4 with no settling steps is DeltaNet's read of tanh keys, and its key memory stores the latest key exactly.
    kc = KeyClean(4, 3, steps=0)
    kc.g.weight.zero_(); kc.g.bias.fill_(50.0); kc.lam_logit.fill_(50.0)
    m, c = kc.memories(x)
    kl = torch.tanh(kc.k(x[:, 1]))
    assert torch.allclose((c @ kl[..., None])[..., 0], kl, atol=1e-4), "the key memory is not a projection"
    kc.o = nn.Identity()
    ref = (m @ F.normalize(torch.tanh(kc.q(x[:, -1])), dim=-1)[..., None])[..., 0]
    assert torch.allclose(kc(x), ref), "v4 with no steps is not DeltaNet's read"
    kn = F.normalize(kl, dim=-1)
    assert torch.allclose((m @ kn[..., None])[..., 0], kc.v(x[:, 1]), atol=1e-4), "v4's value memory is not a delta write"

# With binary keys the stored keys are corners of the cube, yet the gradient still flows through tanh.
    kb = KeyClean(4, 3, binary=True)
    kb.g.weight.zero_(); kb.g.bias.fill_(50.0); kb.lam_logit.fill_(50.0)
    _, cb = kb.memories(x)
    kbl = torch.sign(torch.tanh(kb.k(x[:, 1])))
    assert torch.allclose((cb @ kbl[..., None])[..., 0], kbl, atol=1e-4), "the stored keys are not binary corners"
with torch.enable_grad():
    kb.k.weight.grad = None
    kb.memories(x)[1].sum().backward()
    assert kb.k.weight.grad is not None and kb.k.weight.grad.abs().sum() > 0, "no gradient reaches the binary keys"
print("checks passed")

checks passed


## The Run

As pre-registered: Adam, learning rate 3e-3, 1000 steps of 256 fresh sequences with cue noise 1.5 (drawn from the seeded global generator), seeds 0 to 2 (weights from
`torch.manual_seed(seed)`, data from generator seed `1000 + seed`), one test set of 10 000 sequences from generator
seed 999. Runs go four at a time in forked processes, one thread each; each is deterministic given its seed.

In [5]:
STEPS, BATCH, LR, TEST_N, SEEDS, TRAIN_SIGMA = 1000, 256, 3e-3, 10000, range(3), 1.5


def run(job):
    name, seed = job
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    model = Block(MODELS[name]())
    model.cue_sigma = TRAIN_SIGMA
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
    gen = torch.Generator().manual_seed(1000 + seed)
    t0 = time.process_time()
    for _ in range(STEPS):
        seq, y = make_batch(gen, BATCH)
        loss = F.cross_entropy(model(seq), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    cpu = time.process_time() - t0
    model.eval()
    model.cue_sigma = 0.0
    with torch.no_grad():
        seq, y = make_batch(torch.Generator().manual_seed(999), TEST_N)
        acc = (model(seq).argmax(-1) == y).float().mean().item()
        extra = {}
        if isinstance(model.mixer, KeyClean) and model.mixer.steps > 0:
            _, x = model.features(seq[:2000])
            _, _, states = model.mixer.settle(x)
            extra = dict(beta=[round(b, 3) for b in model.mixer.log_beta.exp().tolist()],
                         alpha=model.mixer.log_alpha.exp().item(),
                         last_move=(states[-1] - states[-2]).norm(dim=-1).mean().item(),
                         first_move=(states[1] - states[0]).norm(dim=-1).mean().item())
    return name, seed, acc, cpu, math.isfinite(loss.item()), extra, model.state_dict()


jobs = [(n, s) for n in MODELS for s in SEEDS]
t0 = time.time()
with mp.get_context("fork").Pool(4) as pool:
    results = pool.map(run, jobs, chunksize=1)
print(f"{len(results)} runs in {(time.time() - t0) / 60:.1f} min of wall clock")
acc = {n: np.array([r[2] for r in results if r[0] == n]) for n in MODELS}
assert all(r[4] for r in results), "a run ended with a non-finite loss"
for n in MODELS:
    print(f"{n:14s}", " ".join(f"{a:.4f}" for a in acc[n]))

9 runs in 7.2 min of wall clock
delta-wide     0.9976 0.9989 0.9992
keyclean-bin   0.1768 0.9629 0.9174
keyclean-bin-0step 0.9199 0.9617 0.9686


## The Statistic

As pre-registered: per model, the mean over seeds with a 95% t-interval; per comparison, the paired difference per
seed with its 95% t-interval (t = 4.303 at 2 degrees of freedom). This cell is for the clean cue; the noisy cues
follow.

In [6]:
T4 = 4.303


def interval(xs):
    m, h = xs.mean(), T4 * xs.std(ddof=1) / math.sqrt(len(xs))
    return m, m - h, m + h


print("model           mean    95% interval")
for n in MODELS:
    m, lo, hi = interval(acc[n])
    print(f"{n:14s}  {m:.4f}  [{lo:.4f}, {hi:.4f}]")
print()
print("paired difference             mean     95% interval")
for a, b in [("keyclean-bin", "delta-wide"), ("keyclean-bin", "keyclean-bin-0step")]:
    m, lo, hi = interval(acc[a] - acc[b])
    print(f"{a:13s} - {b:13s}  {m:+.4f}  [{lo:+.4f}, {hi:+.4f}]")

model           mean    95% interval
delta-wide      0.9986  [0.9965, 1.0007]
keyclean-bin    0.6857  [-0.4107, 1.7821]
keyclean-bin-0step  0.9501  [0.8846, 1.0155]

paired difference             mean     95% interval
keyclean-bin  - delta-wide     -0.3129  [-1.4072, +0.7814]
keyclean-bin  - keyclean-bin-0step  -0.2644  [-1.2964, +0.7677]


## What the Settling Does

The learned inverse temperature of each settling step and the clamp `alpha`, and how far the state moves in the first and the last
step of settling: a last move much smaller than the first means the dynamics settle rather than oscillate.

In [7]:
for r in results:
    if r[5]:
        e = r[5]
        print(f"{r[0]:12s} seed {r[1]}  beta {e['beta']}  alpha {e['alpha']:.3f}  "
              f"first move {e['first_move']:.3f}  last move {e['last_move']:.4f}")

keyclean-bin seed 0  beta [0.74, 0.653, 0.55, 0.446, 0.376]  alpha 1.070  first move 0.585  last move 0.0490
keyclean-bin seed 1  beta [0.441, 0.368, 0.268, 0.161, 0.08]  alpha 0.376  first move 1.955  last move 0.0602
keyclean-bin seed 2  beta [0.406, 0.333, 0.239, 0.143, 0.07]  alpha 0.359  first move 2.133  last move 0.0518


## The Costs

Parameters; memory cells held per sequence (the softmax head's cache grows with the sequence, here 33 positions);
multiply-adds per token to write the memory (the delta rule's read-before-write doubles it) and per query to read it, counting the memory operations only (the
projections cost the same in every model, `2 * 64` inputs to each).

In [8]:
T_CTX = 2 * NPAIR + 1
costs = {}
for n in MODELS:
    torch.manual_seed(0)
    mod = Block(MODELS[n]())
    params = sum(p.numel() for p in mod.parameters() if p.requires_grad)
    mx = mod.mixer
    if isinstance(mx, KeyClean):
        cells = mx.dk * mx.dv + mx.dk ** 2
        write, read = 2 * cells, mx.steps * mx.dk ** 2 + mx.dk * mx.dv
    elif isinstance(mx, HebbHopfield2):
        cells, write, read = mx.h * mx.d ** 2, 2 * mx.h * mx.d ** 2, mx.steps * mx.h * mx.d ** 2
    elif isinstance(mx, Delta):
        cells, write, read = mx.dk * mx.dv, 2 * mx.dk * mx.dv, mx.dk * mx.dv
    elif isinstance(mx, HebbHopfield):
        cells, write, read = mx.d ** 2, mx.d ** 2, mx.steps * mx.d ** 2
    elif isinstance(mx, Hebbian):
        cells, write, read = mx.dk * mx.dv, mx.dk * mx.dv, mx.dk * mx.dv
    elif isinstance(mx, Softmax):
        cells, write, read = T_CTX * (mx.dk + mx.dv), 0, T_CTX * (mx.dk + mx.dv)
    else:
        cells, write, read = 0, 0, 0
    costs[n] = (params, cells, write, read)
    cpu = np.mean([r[3] for r in results if r[0] == n])
    print(f"{n:14s} params {params:6d}  cells {cells:5d}  write/token {write:5d}  read/query {read:5d}  "
          f"training CPU {cpu:.0f} s")

delta-wide     params  63330  cells  1024  write/token  2048  read/query  1024  training CPU 155 s
keyclean-bin   params  53016  cells   960  write/token  1920  read/query  3264  training CPU 203 s
keyclean-bin-0step params  53012  cells   960  write/token  1920  read/query   384  training CPU 129 s


## Recall From a Noisy Cue

The test itself. The trained models, frozen, answer the same 10 000 test sequences (generator seed 999) with the query
key's embedding blurred by noise of standard deviation 0, 1, 1.5 (the trained level), 2 and 2.5 per dimension; the
noise for level `i` comes from generator seed `3000 + i`, re-seeded for every model, so every model hears the same
noisy cues.

In [9]:
SIGMAS = (0.0, 1.0, 1.5, 2.0, 2.5)
seq, y = make_batch(torch.Generator().manual_seed(999), TEST_N)
noisy_acc = {}
for name, seed, *_, state in results:
    torch.manual_seed(0)
    model = Block(MODELS[name]())
    model.load_state_dict(state)
    model.eval()
    with torch.no_grad():
        for i, sigma in enumerate(SIGMAS):
            model.cue_sigma, model.cue_gen = sigma, torch.Generator().manual_seed(3000 + i)
            noisy_acc.setdefault((name, sigma), []).append((model(seq).argmax(-1) == y).float().mean().item())
noisy_acc = {k: np.array(v) for k, v in noisy_acc.items()}
print("model        " + "".join(f"  sigma {s:3.1f}            " for s in SIGMAS))
for name in MODELS:
    row = ""
    for s in SIGMAS:
        m, lo, hi = interval(noisy_acc[(name, s)])
        row += f"  {m:.3f} [{lo:.3f}, {hi:.3f}]"
    print(f"{name:11s}{row}")
print()
for a, b in [("keyclean-bin", "delta-wide"), ("keyclean-bin", "keyclean-bin-0step")]:
    row = ""
    for s in SIGMAS:
        m, lo, hi = interval(noisy_acc[(a, s)] - noisy_acc[(b, s)])
        row += f"  {m:+.3f} [{lo:+.3f}, {hi:+.3f}]"
    print(f"{a} - {b}:{row}")

model          sigma 0.0              sigma 1.0              sigma 1.5              sigma 2.0              sigma 2.5            
delta-wide   0.999 [0.996, 1.001]  0.979 [0.970, 0.988]  0.915 [0.885, 0.945]  0.804 [0.772, 0.835]  0.682 [0.645, 0.719]
keyclean-bin  0.686 [-0.411, 1.782]  0.656 [-0.392, 1.703]  0.580 [-0.308, 1.468]  0.496 [-0.217, 1.208]  0.405 [-0.129, 0.939]
keyclean-bin-0step  0.950 [0.885, 1.016]  0.903 [0.849, 0.957]  0.796 [0.760, 0.833]  0.659 [0.614, 0.704]  0.530 [0.488, 0.572]

keyclean-bin - delta-wide:  -0.313 [-1.407, +0.781]  -0.323 [-1.364, +0.717]  -0.335 [-1.194, +0.523]  -0.308 [-0.990, +0.374]  -0.277 [-0.774, +0.221]
keyclean-bin - keyclean-bin-0step:  -0.264 [-1.296, +0.768]  -0.248 [-1.244, +0.748]  -0.217 [-1.069, +0.635]  -0.163 [-0.830, +0.504]  -0.125 [-0.617, +0.368]


## The Figures

The first: each seed's test accuracy at 16 pairs with a clean cue (dots), the mean and its 95% interval (bars), and chance (dashed
line). The second: accuracy against the noise on the cue, the mean and its 95% interval over seeds, with the trained
level dotted.

In [10]:
plt.rcParams["svg.hashsalt"] = "h11"
os.makedirs("../docs/figures", exist_ok=True)
names = list(MODELS)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for i, n in enumerate(names):
    m, lo, hi = interval(acc[n])
    ax.errorbar([i], [m], yerr=[[m - lo], [hi - m]], fmt="s", color="black", capsize=4, ms=5)
    ax.scatter(i + np.linspace(-0.15, 0.15, len(acc[n])), acc[n], s=12, color="tab:blue", zorder=3)
ax.axhline(1 / NV, ls="--", color="grey", lw=1)
ax.set_xticks(range(len(names)), names, rotation=25, ha="right")
ax.set_ylabel("test recall accuracy")
ax.set_ylim(0, 1)
ax.set_title("Clean-cue recall after noisy training, 16 pairs, 3 seeds (pilot)")
fig.tight_layout()
fig.savefig("../docs/figures/h11v5_recall.svg", metadata={"Date": None})
plt.close(fig)
print("wrote docs/figures/h11v5_recall.svg")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for name, style in [("delta-wide", "^-"), ("keyclean-bin", "s-"), ("keyclean-bin-0step", "d--")]:
    ms, los, his = zip(*(interval(noisy_acc[(name, s)]) for s in SIGMAS))
    ax.errorbar(SIGMAS, ms, yerr=[np.subtract(ms, los), np.subtract(his, ms)], fmt=style, capsize=3, ms=4, label=name)
ax.axvline(TRAIN_SIGMA, ls=":", color="grey", lw=1)
ax.axhline(1 / NV, ls="--", color="grey", lw=1)
ax.set_xlabel("noise on the cue, per dimension (trained at 1.5)")
ax.set_ylabel("test recall accuracy")
ax.set_ylim(0, 1)
ax.set_title("Recall from a noisy cue, 16 pairs, 3 seeds (pilot)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("../docs/figures/h11v5_noise.svg", metadata={"Date": None})
plt.close(fig)
print("wrote docs/figures/h11v5_noise.svg")

wrote docs/figures/h11v5_recall.svg


wrote docs/figures/h11v5_noise.svg
